In [2]:
import pandas as pd
import json
import glob
from helper import get_metrics_from_cve as get_metrics
import sqlite3

In [3]:
#json to df

rows = []

# Loop over all json files in your raw folder
for filepath in glob.glob("../data/raw/CVE/202[2-5]/**/*.json", recursive=True):
    with open(filepath, "r", encoding="utf-8") as f:
        try:
            data = json.load(f)
            
            # Extract top-level metadata
            meta = data.get("cveMetadata", {})
            cve_id = meta.get("cveId")
            date_pub = meta.get("datePublished")
            state = meta.get("state")
            container = data.get("containers")
            # Extract cna container details
            score,severity,atk_vec ,_ = get_metrics(data.get("containers"))
           
            
            # Extract description
            desc_list = container.get("cna").get("descriptions",[])
            desc = None
            if len(desc_list):
                desc = desc_list[0].get("value")
            

            if cve_id:
                rows.append({
                    "cve_id": cve_id,
                    "date_published": date_pub,
                    "cve_state": state,
                    "score": score,
                    "severity":severity,
                    "attack_vector":atk_vec,
                    "description": desc
                })
            else:
                print(cve_id)
        except Exception as e:
            print(e,filepath)
            continue

# Save as a clean CSV table
df_cve = pd.DataFrame(rows)


In [4]:
# convert data to sql
print(df_cve.head(10))
# print(len(rows))
df_cve.columns

          cve_id            date_published  cve_state  score severity  \
0  CVE-2022-0001  2022-03-11T00:00:00.000Z  PUBLISHED    6.5   MEDIUM   
1  CVE-2022-0002  2022-03-11T17:54:36.000Z  PUBLISHED    6.5   MEDIUM   
2  CVE-2022-0003                       NaN   REJECTED    NaN      NaN   
3  CVE-2022-0004  2022-05-12T16:36:02.000Z  PUBLISHED    6.8   MEDIUM   
4  CVE-2022-0005  2022-05-12T16:36:04.000Z  PUBLISHED    2.4      LOW   
5  CVE-2022-0010  2023-05-22T07:22:51.662Z  PUBLISHED    7.8     HIGH   
6  CVE-2022-0011  2022-02-10T18:10:15.524Z  PUBLISHED    6.5   MEDIUM   
7  CVE-2022-0012  2022-01-12T17:30:15.528Z  PUBLISHED    6.1   MEDIUM   
8  CVE-2022-0013  2022-01-12T17:30:17.158Z  PUBLISHED    5.0   MEDIUM   
9  CVE-2022-0014  2022-01-12T17:30:18.718Z  PUBLISHED    6.7   MEDIUM   

  attack_vector                                        description  
0         LOCAL  Non-transparent sharing of branch predictor se...  
1         LOCAL  Non-transparent sharing of branch predict

Index(['cve_id', 'date_published', 'cve_state', 'score', 'severity',
       'attack_vector', 'description'],
      dtype='str')

In [5]:
# csv to df
df_nvd = pd.read_csv("../data/raw/known_exploited_vulnerabilities.csv")
df_nvd.columns

Index(['cveID', 'vendorProject', 'product', 'vulnerabilityName', 'dateAdded',
       'shortDescription', 'requiredAction', 'dueDate',
       'knownRansomwareCampaignUse', 'forensicTriage', 'notes', 'cwes'],
      dtype='str')

In [ ]:
#df to sql
conn = sqlite3.connect(':memory:')
df_cve.to_sql('nvd_cves', conn, index=False, if_exists='replace')     
df_nvd.to_sql('cisa_kev', conn, index=False, if_exists='replace')

cursor = conn.cursor()
cursor.execute("DROP TABLE IF EXISTS cisa_nvd_joined;")
cursor.execute("""
    CREATE TABLE cisa_nvd_joined AS
    SELECT 
        -- Common Key (from CISA)
        c.cveID AS cve_id,
        
        -- CISA KEV Columns (from table 'cisa_kev', alias 'c')
        c.vendorProject,
        c.product,
        c.vulnerabilityName,
        c.dateAdded AS cisa_date_added,
        c.shortDescription AS cisa_description,
        c.requiredAction,
        c.dueDate,
        c.knownRansomwareCampaignUse,
        c.forensicTriage,
        c.notes,
        c.cwes,
        
        -- NVD Columns (from table 'nvd_cves', alias 'n')
        n.date_published AS nvd_date_published,
        n.cve_state,
        n.score AS cvss_score,
        n.severity AS cvss_severity,
        n.attack_vector,
        n.description AS nvd_description

    FROM cisa_kev c
    INNER JOIN nvd_cves n 
        ON UPPER(TRIM(c.cveID)) = UPPER(TRIM(n.cve_id));
""")

joined_df = pd.read_sql_query("SELECT * FROM cisa_nvd_joined", conn)





In [7]:

print(f"Pre-Join CISA Count: {len(df_nvd)}")  
print(f"Pre-Join NVD Count:  {len(df_cve)}")  
print(f"Post-Join SQL Count: {len(joined_df)}")

match_rate = (len(joined_df) / len(df_nvd)) * 100
print(f"Overall Match Rate:  {match_rate:.2f}%")

print("\nJoined Dataset Sample:")
print(joined_df.head())

Pre-Join CISA Count: 1716
Pre-Join NVD Count:  143544
Post-Join SQL Count: 649
Overall Match Rate:  37.82%

Joined Dataset Sample:
           cve_id vendorProject            product  \
0  CVE-2025-39964         Linux             Kernel   
1  CVE-2025-39682         Linux             Kernel   
2  CVE-2025-25249      Fortinet  Multiple Products   
3  CVE-2023-49105      ownCloud           ownCloud   
4   CVE-2022-0995         Linux             Kernel   

                                   vulnerabilityName cisa_date_added  \
0          Linux Kernel Race Condition Vulnerability      2026-09-18   
1  Linux Kernel Improper Check for Unusual or Exc...      2026-09-18   
2  Fortinet Multiple Products Heap-based Buffer O...      2026-09-09   
3     ownCloud Improper Authentication Vulnerability      2026-08-27   
4     Linux Kernel Out-of-Bounds Write Vulnerability      2026-08-26   

                                    cisa_description  \
0  Linux Kernel contains a race condition vulnera...   

In [ ]:
unmatched_cisa_sql = """
    SELECT 
        c.cveID,
        c.vendorProject,
        c.product,
        c.vulnerabilityName,
        c.dateAdded
    FROM cisa_kev c
    LEFT JOIN nvd_cves n 
        ON UPPER(TRIM(c.cveID)) = UPPER(TRIM(n.cve_id))
    WHERE n.cve_id IS NULL;
"""

df_cve.to_sql('nvd_cves', conn, index=False, if_exists='replace')
df_nvd.to_sql('cisa_kev', conn, index=False, if_exists='replace')

unmatched_df = pd.read_sql_query(unmatched_cisa_sql, conn)

print(f"Total Unmatched CISA Rows: {len(unmatched_df)}")
print("\nFirst 10 Unmatched Vulnerabilities:")
print(unmatched_df.head(10))

Total Unmatched CISA Rows: 1067

First 10 Unmatched Vulnerabilities:
            cveID vendorProject                                   product  \
0  CVE-2026-53266         Linux                                    Kernel   
1  CVE-2026-58704        Google                                     Pixel   
2  CVE-2026-76460         Cisco                  Identity Services Engine   
3  CVE-2026-87886       Acronis                                    Backup   
4  CVE-2026-76461         Cisco                      Secure Email Gateway   
5  CVE-2026-84869   ConnectWise                             ScreenConnect   
6  CVE-2026-42016         JFrog                               Artifactory   
7  CVE-2026-42018         JFrog                               Artifactory   
8  CVE-2026-85706        GitLab  Community Edition and Enterprise Edition   
9  CVE-2026-86060      MikroTik                                  RouterOS   

                                   vulnerabilityName   dateAdded  
0     Linux Kern

In [34]:
known_df= joined_df[joined_df["knownRansomwareCampaignUse"]!="Unknown"]
# print(known_df)
known_df
known_df["attack_vector"].value_counts() 

attack_vector
NETWORK             87
LOCAL                5
ADJACENT_NETWORK     2
Name: count, dtype: int64